In [15]:
# ============================================================
# KHỞI TẠO: Import module & nạp dữ liệu
# ============================================================
# data_quality.py nằm cùng thư mục với notebook này
# (data/quality_checks/), nên import trực tiếp, không cần sys.path.

import pandas as pd
import numpy as np

from data_quality import (
    audit,
    check_overview,
    check_plausibility,
    check_distribution,
    check_hidden_issues,
    check_logic,
    check_bmi_missing,
)

# Đường dẫn file cần kiểm tra (tính từ data/quality_checks/)
patient_input = "../processed/patient_clean.csv"
df = pd.read_csv(patient_input)

print("Đã nạp:", df.shape)
print(df.head())

Đã nạp: (2326, 5)
   patientunitstayid   age  male_sex        bmi  mortality
0             141764  87.0       0.0        NaN          0
1             141765  87.0       0.0  18.745276          0
2             143870  76.0       1.0  27.788734          0
3             144815  34.0       0.0  20.217741          0
4             145427  61.0       1.0  29.007201          0


In [16]:
# ============================================================
# KIỂM TRA CHUNG: dùng module data_quality
# ============================================================
# Hàm audit() chạy tất cả các kiểm tra tổng quát có thể áp dụng
# cho BẤT KỲ bảng dữ liệu nào (patient, vital, lab, apache).
# Với patient_clean.csv, ta truyền thêm plausibility_rules phù hợp.

PATIENT_RULES = {
    "age":       (18, 120, "tuổi"),
    "male_sex":  (0, 1,    "0=Female, 1=Male"),
    "bmi":       (10, 80,  "kg/m²"),
    "mortality": (0, 1,    "0=Alive, 1=Expired"),
}

audit(
    df,
    name="patient_clean.csv",
    label_col="mortality",
    plausibility_rules=PATIENT_RULES,
)


TỔNG QUAN: patient_clean.csv
Kích thước : 2,326 dòng × 5 cột
Bộ nhớ     : 91.0 KB

Kiểu dữ liệu:
patientunitstayid      int64
age                  float64
male_sex             float64
bmi                  float64
mortality              int64

Missing values:
          missing   pct
male_sex        2  0.09
bmi           169  7.27

Số dòng trùng lặp hoàn toàn: 0

KIỂM TRA PLAUSIBILITY (ngưỡng sinh lý)
✅ age             | min=     18.00 max=     90.00 | ngoài [18, 120] (tuổi): 0 dòng
✅ male_sex        | min=      0.00 max=      1.00 | ngoài [0, 1] (0=Female, 1=Male): 0 dòng
✅ bmi             | min=     12.55 max=     79.98 | ngoài [10, 80] (kg/m²): 0 dòng
✅ mortality       | min=      0.00 max=      1.00 | ngoài [0, 1] (0=Alive, 1=Expired): 0 dòng

PHÂN BỐ CÁC CỘT SỐ

--- patientunitstayid ---
  count=2,326  mean=1698311.20  std=1017669.72
  min=141764.00  q25=827085.25  median=1630112.00  q75=2716246.25  max=3353113.00

--- age ---
  count=2,326  mean=63.37  std=17.68
  min=18.00  q25=5

In [17]:
# ============================================================
# KIỂM TRA RIÊNG: Cohort bệnh nhân
# ============================================================
# Những kiểm tra đặc thù chỉ áp dụng cho bảng patient,
# không có trong module chung.

print("=" * 60)
print("KIỂM TRA RIÊNG: COHORT BỆNH NHÂN")
print("=" * 60)

# 1. Tuổi: phải >= 18 (đã lọc cohort)
print(f"\n1. Tuổi:")
print(f"   min={df['age'].min():.0f}  max={df['age'].max():.0f}  "
      f"median={df['age'].median():.0f}")
n_under18 = (df["age"] < 18).sum()
print(f"   {'✅' if n_under18 == 0 else '⚠️'} Số bệnh nhân < 18 tuổi: {n_under18}")

# 2. Giới tính
print(f"\n2. Giới tính (male_sex):")
n_male   = (df["male_sex"] == 1).sum()
n_female = (df["male_sex"] == 0).sum()
n_unknown = df["male_sex"].isna().sum()
print(f"   Nam   : {n_male:,} ({n_male/len(df)*100:.1f}%)")
print(f"   Nữ    : {n_female:,} ({n_female/len(df)*100:.1f}%)")
print(f"   Unknown: {n_unknown} ({n_unknown/len(df)*100:.2f}%)")
if n_unknown > 0:
    print(f"   ℹ️ {n_unknown} dòng male_sex=NaN — XGBoost tự xử lý, không cần bỏ")

KIỂM TRA RIÊNG: COHORT BỆNH NHÂN

1. Tuổi:
   min=18  max=90  median=66
   ✅ Số bệnh nhân < 18 tuổi: 0

2. Giới tính (male_sex):
   Nam   : 1,402 (60.3%)
   Nữ    : 922 (39.6%)
   Unknown: 2 (0.09%)
   ℹ️ 2 dòng male_sex=NaN — XGBoost tự xử lý, không cần bỏ


In [18]:
# ============================================================
# KIỂM TRA RIÊNG: Phân bố BMI
# ============================================================
# BMI là đặc trưng quan trọng, cần kiểm tra kỹ:
#   - Giá trị bất thường (đã lọc nhưng cần xác nhận)
#   - Tỷ lệ NaN
#   - Phân bố có khớp bài báo gốc không (28.6 ± 8.1)

print("=" * 60)
print("KIỂM TRA RIÊNG: PHÂN BỐ BMI")
print("=" * 60)

bmi = df["bmi"].dropna()
print(f"\ncount  = {len(bmi):,} / {len(df):,} ({len(bmi)/len(df)*100:.1f}%)")
print(f"mean   = {bmi.mean():.2f}   (bài báo gốc: 28.6)")
print(f"std    = {bmi.std():.2f}    (bài báo gốc: 8.1)")
print(f"min    = {bmi.min():.2f}")
print(f"q25    = {bmi.quantile(.25):.2f}")
print(f"median = {bmi.median():.2f}")
print(f"q75    = {bmi.quantile(.75):.2f}")
print(f"max    = {bmi.max():.2f}")

# Kiểm tra outlier còn sót
n_outlier = ((bmi < 10) | (bmi > 80)).sum()
print(f"\n{'✅' if n_outlier == 0 else '⚠️'} BMI ngoài [10, 80]: {n_outlier} dòng")

# So sánh với bài báo
diff_mean = abs(bmi.mean() - 28.6)
diff_std  = abs(bmi.std() - 8.1)
print(f"\nSo với bài báo gốc:")
print(f"   |mean - 28.6| = {diff_mean:.2f}  {'✅' if diff_mean < 1 else '⚠️'}")
print(f"   |std  -  8.1| = {diff_std:.2f}  {'✅' if diff_std < 1 else '⚠️'}")

# Kiểm tra phân bố lệch
skew = bmi.skew()
print(f"\nSkewness = {skew:.2f}  "
      f"{'✅ (gần đối xứng)' if abs(skew) < 1 else '⚠️ (lệch)'}")

KIỂM TRA RIÊNG: PHÂN BỐ BMI

count  = 2,157 / 2,326 (92.7%)
mean   = 28.57   (bài báo gốc: 28.6)
std    = 8.02    (bài báo gốc: 8.1)
min    = 12.55
q25    = 23.08
median = 27.26
q75    = 32.18
max    = 79.98

✅ BMI ngoài [10, 80]: 0 dòng

So với bài báo gốc:
   |mean - 28.6| = 0.03  ✅
   |std  -  8.1| = 0.08  ✅

Skewness = 1.40  ⚠️ (lệch)


In [19]:
# ============================================================
# KIỂM TRA RIÊNG: Mất cân bằng nhãn & scale_pos_weight
# ============================================================
# Với bài toán tử vong ICU, lớp 1 (tử vong) thường là thiểu số.
# XGBoost cần scale_pos_weight để chú ý hơn đến lớp thiểu số.

print("=" * 60)
print("KIỂM TRA RIÊNG: MẤT CÂN BẰNG NHÃN")
print("=" * 60)

vc = df["mortality"].value_counts()
n_alive   = vc.get(0, 0)
n_expired = vc.get(1, 0)

print(f"\nAlive   (0): {n_alive:,} ({n_alive/len(df)*100:.2f}%)")
print(f"Expired (1): {n_expired:,} ({n_expired/len(df)*100:.2f}%)")
print(f"Tỷ lệ    : {n_alive/n_expired:.2f} : 1")

# scale_pos_weight = số mẫu âm / số mẫu dương
spw = n_alive / n_expired
print(f"\n💡 Gợi ý scale_pos_weight khi train XGBoost: {spw:.2f}")
print(f"   (dùng trong XGBClassifier(scale_pos_weight={spw:.2f}))")

KIỂM TRA RIÊNG: MẤT CÂN BẰNG NHÃN

Alive   (0): 2,135 (91.79%)
Expired (1): 191 (8.21%)
Tỷ lệ    : 11.18 : 1

💡 Gợi ý scale_pos_weight khi train XGBoost: 11.18
   (dùng trong XGBClassifier(scale_pos_weight=11.18))


In [20]:
# ============================================================
# KIỂM TRA RIÊNG: Khóa chính
# ============================================================
# patientunitstayid phải là duy nhất trong bảng patient_clean
# vì mỗi dòng = 1 đợt nằm ICU.

print("=" * 60)
print("KIỂM TRA RIÊNG: KHÓA CHÍNH")
print("=" * 60)

n_rows   = len(df)
n_unique = df["patientunitstayid"].nunique()
n_dup    = n_rows - n_unique

print(f"\nSố dòng              : {n_rows:,}")
print(f"Số patientunitstayid : {n_unique:,}")
print(f"Số trùng lặp         : {n_dup}")
print(f"{'✅' if n_dup == 0 else '⚠️'} Khóa chính duy nhất")

# Kiểm tra ID dương
n_bad_id = (df["patientunitstayid"] <= 0).sum()
print(f"{'✅' if n_bad_id == 0 else '⚠️'} patientunitstayid <= 0: {n_bad_id} dòng")

# Xem mẫu ID
print(f"\nMẫu 5 ID: {df['patientunitstayid'].head().tolist()}")

KIỂM TRA RIÊNG: KHÓA CHÍNH

Số dòng              : 2,326
Số patientunitstayid : 2,326
Số trùng lặp         : 0
✅ Khóa chính duy nhất
✅ patientunitstayid <= 0: 0 dòng

Mẫu 5 ID: [141764, 141765, 143870, 144815, 145427]


In [21]:
# ============================================================
# KIỂM TRA RIÊNG: Tổng hợp NaN & quyết định
# ============================================================
# Tổng hợp tất cả NaN trong bảng, đánh giá mức độ ảnh hưởng.

print("=" * 60)
print("KIỂM TRA RIÊNG: TỔNG HỢP NaN")
print("=" * 60)

miss = df.isna().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_df = pd.DataFrame({
    "missing": miss,
    "pct": miss_pct,
    "quyết định": ""
})

for col in miss_df.index:
    if miss_df.loc[col, "missing"] == 0:
        miss_df.loc[col, "quyết định"] = "Không có NaN"
    elif miss_df.loc[col, "pct"] < 10:
        miss_df.loc[col, "quyết định"] = "Giữ NaN (XGBoost xử lý)"
    elif miss_df.loc[col, "pct"] < 30:
        miss_df.loc[col, "quyết định"] = "Giữ NaN, cân nhắc impute"
    else:
        miss_df.loc[col, "quyết định"] = "⚠️ Cân nhắc loại bỏ cột"

print(miss_df.to_string())

print(f"\nTổng NaN toàn bảng: {miss.sum():,} ô "
      f"({miss.sum() / (len(df) * len(df.columns)) * 100:.2f}%)")

KIỂM TRA RIÊNG: TỔNG HỢP NaN
                   missing   pct               quyết định
patientunitstayid        0  0.00             Không có NaN
age                      0  0.00             Không có NaN
male_sex                 2  0.09  Giữ NaN (XGBoost xử lý)
bmi                    169  7.27  Giữ NaN (XGBoost xử lý)
mortality                0  0.00             Không có NaN

Tổng NaN toàn bảng: 171 ô (1.47%)


In [22]:
# ============================================================
# KIỂM TRA RIÊNG: Phân bố tuổi theo nhãn
# ============================================================
# Kiểm tra xem tuổi có phân biệt được nhóm tử vong không
# (dấu hiệu cho thấy đặc trưng có ích).

print("=" * 60)
print("KIỂM TRA RIÊNG: TUỔI THEO NHÓM TỬ VONG")
print("=" * 60)

print("\nThống kê tuổi theo mortality:")
print(df.groupby("mortality")["age"].agg(
    count="count", mean="mean", median="median",
    min="min", max="max"
).round(2).to_string())

# So sánh trung bình
age_alive   = df.loc[df["mortality"] == 0, "age"].mean()
age_expired = df.loc[df["mortality"] == 1, "age"].mean()
print(f"\nTuổi TB - Alive  : {age_alive:.1f}")
print(f"Tuổi TB - Expired: {age_expired:.1f}")
print(f"Chênh lệch       : {age_expired - age_alive:+.1f} tuổi")
if age_expired > age_alive:
    print("→ ✅ Nhóm tử vong có tuổi TB cao hơn — tuổi là đặc trưng có ích")
else:
    print("→ ℹ️ Nhóm tử vong không cao tuổi hơn rõ rệt")

KIỂM TRA RIÊNG: TUỔI THEO NHÓM TỬ VONG

Thống kê tuổi theo mortality:
           count   mean  median   min   max
mortality                                  
0           2135  62.49    65.0  18.0  90.0
1            191  73.22    76.0  28.0  90.0

Tuổi TB - Alive  : 62.5
Tuổi TB - Expired: 73.2
Chênh lệch       : +10.7 tuổi
→ ✅ Nhóm tử vong có tuổi TB cao hơn — tuổi là đặc trưng có ích


In [23]:
# ============================================================
# KIỂM TRA RIÊNG: Tương quan giữa các đặc trưng
# ============================================================
# Tương quan quá cao (> 0.9) giữa 2 đặc trưng → dư thừa,
# có thể loại bớt 1 cột. Ở đây chỉ có 3 đặc trưng số nên ít lo.

print("=" * 60)
print("KIỂM TRA RIÊNG: TƯƠNG QUAN")
print("=" * 60)

num_cols = ["age", "male_sex", "bmi", "mortality"]
corr = df[num_cols].corr().round(3)
print("\nMa trận tương quan:")
print(corr.to_string())

# Tìm cặp tương quan cao (bỏ qua đường chéo và cặp với mortality)
print("\nCác cặp tương quan |r| > 0.5 (không tính mortality):")
found = False
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):
        c1, c2 = num_cols[i], num_cols[j]
        if c2 == "mortality":
            continue
        r = corr.loc[c1, c2]
        if abs(r) > 0.5:
            print(f"  ⚠️ {c1} ↔ {c2}: r = {r:.3f}")
            found = True
if not found:
    print("  ✅ Không có cặp tương quan cao")

# Tương quan với nhãn
print("\nTương quan với mortality (sắp xếp giảm dần):")
print(corr["mortality"].drop("mortality").sort_values(ascending=False).to_string())

KIỂM TRA RIÊNG: TƯƠNG QUAN

Ma trận tương quan:
             age  male_sex    bmi  mortality
age        1.000    -0.071 -0.060      0.167
male_sex  -0.071     1.000 -0.011     -0.017
bmi       -0.060    -0.011  1.000     -0.026
mortality  0.167    -0.017 -0.026      1.000

Các cặp tương quan |r| > 0.5 (không tính mortality):
  ✅ Không có cặp tương quan cao

Tương quan với mortality (sắp xếp giảm dần):
age         0.167
male_sex   -0.017
bmi        -0.026


In [24]:
# ============================================================
# KẾT LUẬN: Checklist chất lượng dữ liệu
# ============================================================
# Tổng hợp tất cả kiểm tra thành checklist Pass/Fail.

print("=" * 60)
print("CHECKLIST CHẤT LƯỢNG: patient_clean.csv")
print("=" * 60)

checks = {
    "Khóa chính duy nhất":           df["patientunitstayid"].duplicated().sum() == 0,
    "Không có dòng trùng hoàn toàn": df.duplicated().sum() == 0,
    "age >= 18":                     (df["age"] < 18).sum() == 0,
    "age không NaN":                 df["age"].isna().sum() == 0,
    "mortality chỉ 0/1":             df["mortality"].dropna().isin([0, 1]).all(),
    "mortality không NaN":           df["mortality"].isna().sum() == 0,
    "male_sex chỉ 0/1 hoặc NaN":     df["male_sex"].dropna().isin([0, 1]).all(),
    "BMI trong [10, 80]":            ((df["bmi"].dropna() >= 10) & (df["bmi"].dropna() <= 80)).all(),
    "BMI NaN < 10%":                 df["bmi"].isna().mean() < 0.10,
    "Có đủ 2 lớp mortality":         df["mortality"].nunique() == 2,
}

for name, passed in checks.items():
    print(f"{'✅' if passed else '❌'} {name}")

n_pass = sum(checks.values())
n_total = len(checks)
print(f"\nKết quả: {n_pass}/{n_total} kiểm tra đạt")

if n_pass == n_total:
    print("\n🎉 DỮ LIỆU SẴN SÀNG cho bước tiếp theo (merge & train)!")
else:
    print(f"\n⚠️ Còn {n_total - n_pass} vấn đề cần xử lý trước khi train.")

CHECKLIST CHẤT LƯỢNG: patient_clean.csv
✅ Khóa chính duy nhất
✅ Không có dòng trùng hoàn toàn
✅ age >= 18
✅ age không NaN
✅ mortality chỉ 0/1
✅ mortality không NaN
✅ male_sex chỉ 0/1 hoặc NaN
✅ BMI trong [10, 80]
✅ BMI NaN < 10%
✅ Có đủ 2 lớp mortality

Kết quả: 10/10 kiểm tra đạt

🎉 DỮ LIỆU SẴN SÀNG cho bước tiếp theo (merge & train)!
